# Singular vs Non-Singular Matrices

*Course notes for **Math for Machine Learning**, C1 · W1 · L1 — "Singular vs Non-Singular Matrices" (DeepLearning.AI).*

In [the geometric notion of singularity](./C1_W1_L1_geometric_notion_of_singularity.ipynb) we saw that singularity is about the *lines' directions*, not the constants. This lecture makes that precise: we strip a system down to its **coefficient matrix** and call **that matrix** singular or non-singular.

> A **matrix** is **non-singular** if the system with all constants set to **0** has only the trivial solution (everything zero); it is **singular** if that system has infinitely many solutions.

And the headline result: **constants don't matter for singularity** — only the coefficients do.

In [1]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

def classify(A):
    """Classify a square coefficient matrix as singular / non-singular."""
    A = np.asarray(A, dtype=float)
    det = np.linalg.det(A)
    rank = np.linalg.matrix_rank(A)
    kind = "non-singular" if abs(det) > 1e-9 else "singular"
    return det, rank, kind

## 1. A system is really just its matrix

Take two 2×2 systems (constants set to 0 to focus on the coefficients):

$$
\textbf{System 1}\;\begin{cases} a + b = 0 \\ a + 2b = 0 \end{cases} \;\Rightarrow\; \begin{bmatrix} 1 & 1 \\ 1 & 2 \end{bmatrix}
\qquad\qquad
\textbf{System 2}\;\begin{cases} a + b = 0 \\ 2a + 2b = 0 \end{cases} \;\Rightarrow\; \begin{bmatrix} 1 & 1 \\ 2 & 2 \end{bmatrix}
$$

We keep only the table of coefficients — the **matrix**. System 1's matrix has independent rows (unique solution → **non-singular**); System 2's second row is twice the first (infinitely many solutions → **singular**).

In [2]:
for name, A in {"System 1": [[1, 1], [1, 2]],
                "System 2": [[1, 1], [2, 2]]}.items():
    det, rank, kind = classify(A)
    print(f"{name}: matrix={np.array(A).tolist()}  det={det:5.1f}  rank={rank}  ->  {kind}")

System 1: matrix=[[1, 1], [1, 2]]  det=  1.0  rank=2  ->  non-singular
System 2: matrix=[[1, 1], [2, 2]]  det=  0.0  rank=1  ->  singular


## 2. Constants don't matter for singularity (3×3)

Four 3×3 systems in $a, b, c$. The right-hand sides differ, but singularity is decided purely by the **left-hand coefficients**:

$$
\textbf{1}\begin{cases}a+b+c=10\\a+2b+c=15\\a+b+2c=12\end{cases}\;
\textbf{2}\begin{cases}a+b+c=10\\a+b+2c=15\\a+b+3c=20\end{cases}\;
\textbf{3}\begin{cases}a+b+c=10\\a+b+2c=15\\a+b+3c=18\end{cases}\;
\textbf{4}\begin{cases}a+b+c=10\\2a+2b+2c=15\\3a+3b+3c=20\end{cases}
$$

- **System 1** — three independent rows → **non-singular**.
- **System 2 & 3** — identical coefficient matrix; the $a$ and $b$ columns are the same, rows aren't independent → **singular** (note systems 2 and 3 differ *only* in constants, yet both are singular).
- **System 4** — rows are multiples of each other ($\times 1, \times 2, \times 3$) → **singular**.

In [3]:
systems = {
    "System 1": [[1, 1, 1], [1, 2, 1], [1, 1, 2]],
    "System 2": [[1, 1, 1], [1, 1, 2], [1, 1, 3]],
    "System 3": [[1, 1, 1], [1, 1, 2], [1, 1, 3]],  # same coefficients as System 2
    "System 4": [[1, 1, 1], [2, 2, 2], [3, 3, 3]],
}
for name, A in systems.items():
    det, rank, kind = classify(A)
    print(f"{name}: det={det:6.2f}  rank={rank}/3  ->  {kind}")

System 1: det=  1.00  rank=3/3  ->  non-singular
System 2: det=  0.00  rank=2/3  ->  singular
System 3: det=  0.00  rank=2/3  ->  singular
System 4: det=  0.00  rank=1/3  ->  singular


## 3. Why "set the constants to 0" is the right test

Setting the right-hand side to **0** gives a *homogeneous* system $A\mathbf{x} = \mathbf{0}$. The all-zero vector is always a solution, so the only question is whether it is the **only** one:

- **Non-singular** ($A$ full rank): the trivial solution $a=b=c=0$ is unique → *complete*.
- **Singular** ($A$ rank-deficient): there are non-zero solutions too → infinitely many.

Let's see this for System 1 (non-singular) and System 2 (singular), with constants zeroed.

In [4]:
A1 = np.array(systems["System 1"], dtype=float)
A2 = np.array(systems["System 2"], dtype=float)
zero = np.zeros(3)

# Non-singular: solve A1 x = 0  ->  unique trivial solution
print("System 1 (non-singular), A x = 0  ->  x =", np.linalg.solve(A1, zero))
print("  unique solution a=b=c=0  (complete)\n")

# Singular: A2 x = 0 has non-trivial solutions; the null space spans them.
# From a+b+c=0, a+b+2c=0, a+b+3c=0  ->  c=0 and a = -b  (e.g. a=1, b=-1, c=0).
x_nontrivial = np.array([1.0, -1.0, 0.0])
print("System 2 (singular), a non-zero solution x =", x_nontrivial)
print("  check A x =", A2 @ x_nontrivial, " (= 0, so infinitely many solutions exist)")

System 1 (non-singular), A x = 0  ->  x = [0. 0. 0.]
  unique solution a=b=c=0  (complete)

System 2 (singular), a non-zero solution x = [ 1. -1.  0.]
  check A x = [0. 0. 0.]  (= 0, so infinitely many solutions exist)


## 4. Takeaways

- Drop the constants and a system becomes a **coefficient matrix** — singularity is a property of *that matrix*.
- **Constants don't matter:** systems 2 and 3 share a matrix and are both singular despite different right-hand sides.
- **Test:** set the constants to 0 (homogeneous system $A\mathbf{x}=\mathbf{0}$).
  - Only the trivial solution → **non-singular** (full rank, $\det \neq 0$).
  - Non-trivial solutions exist → **singular** (rank-deficient, $\det = 0$).
- Equivalent algebraic signals: $\det(A)=0$, rank $<$ size, linearly dependent rows/columns.

*Next:* the **determinant** itself — how to compute it and why it measures singularity.